In [1]:
# test space for first contraction classification algorithm based on peak finding and features on small data set 

In [16]:
import os.path as op
import mne 
import os
import numpy as np 
import pandas as pd
from matplotlib.backends.backend_tkagg import FigureCanvasTkAgg
from matplotlib import pyplot as plt
import tkinter as tk
import glob
import matplotlib
from scipy.fft import fft, ifft,fftfreq
from scipy.signal import welch, find_peaks 

matplotlib.use('QtAgg') 
mne.set_log_level("CRITICAL")   # Only show warnings and errors - "ERROR" to only show errors and "CRITICAL" to completely surpress 

Defining initial variables 

In [3]:
to_keep=['Fp1', 'Fp2', 'C3', 'Cz', 'C4', 'P3', 'Pz', 'P4', 'O1', 'O2','EOG1','EOG2','Corr', 'Zygo', 'Menton','Trigger']
eeg_ch= ['Fp1', 'Fp2', 'C3', 'Cz', 'C4', 'P3', 'Pz', 'P4', 'O1', 'O2']
emg_ch= ['Corr', 'Zygo', 'Menton'] # Menton is chin EMG for sleep scoring
eog_ch= ['EOG1', 'EOG2']
trigger_ch= ['Trigger']

inter_trigger_length = 30
num_epochs = 60 # 60 epochs for each session 

raw_path= "/Users/zeynepozkaya/Desktop/Consciousness_Research/Python_Scripts/EEG_data"
current_index=0
inter_trigger_length=10
window = 50 
step = 1 

Pre-Processing Steps

In [23]:
# pre-processes data for each subject and returns df with session divided into 60 epochs w meta-data attached 
def pre_process_subjets(subject,block):
    global raw_path
    global frq

    subject_name = subject + block 
    file= op.join(raw_path,'{}.edf'.format(subject_name))
    raw =  mne.io.read_raw_edf(file,preload=True)

    if 'Fp1/F3' in raw.info['ch_names']:
        mne.rename_channels(info=raw.info,mapping={'Fp1/F3':'Fp1' ,'Fp2/F4':'Fp2'})
        
    if '36' in raw.info['ch_names']:
        mne.rename_channels(info=raw.info,mapping={'36':'Corr' ,'37':'Zygo','38':'Menton'})

    if 'E1' in raw.info['ch_names']:
        mne.rename_channels(info=raw.info,mapping={'E1':'EOG1' ,'E2':'EOG2'})
    
    if 'Corru' in raw.info['ch_names']:
        raw.rename_channels({'Corru': 'Corr'})

    raw.set_channel_types(mapping={'Corr':'emg','Zygo':'emg','Menton':'emg','Trigger':'stim','EOG1':'eog','EOG2':'eog'})

    for ch in raw.info['ch_names']:
        if ch not in to_keep:
            raw.drop_channels([ch]) # only keeps channel that contains the trigger (where a stimulus was presented )

    raw= raw.resample(sfreq=250)

    filter_params_emg = {'lpass': 100,'hpass': 10,'notches': [50]}
    raw.filter(l_freq=filter_params_emg['hpass'],h_freq=filter_params_emg['lpass'],picks=emg_ch)
    filter_params_eeg_eog = {'lpass': 70,'hpass': 0.3,'notches': [50]}
    raw.filter(l_freq=filter_params_eeg_eog['hpass'],h_freq=filter_params_eeg_eog['lpass'],picks=eeg_ch+eog_ch)
    raw.notch_filter(filter_params_eeg_eog['notches'], method='fft', picks=emg_ch+eeg_ch+eog_ch)

    frq=raw.info['sfreq']

    # create dataframe based on events 
    events_all= mne.find_events(raw)
    events= mne.pick_events(events_all,include=[201,202])

    events_all= mne.find_events(raw) # Get all the events (triggers) in your EEG data
    events= mne.pick_events(events_all,include=[201,202]) # Pick events of interest (where a stimulus was presented)
    events[0][0] # gives a single sample (where first event is )

    df_triggers=pd.DataFrame(data=events,columns=['Time(Sample)','dunno','Trigger'])
    df_triggers.drop(columns=['dunno'],inplace=True)
    df_triggers['Time(s)']=df_triggers['Time(Sample)']/raw.info['sfreq']


    # incorporating meta_data 
    cols_inc = ["Subject","Nap_ID","Trigger", "Expected_Muscle","Nb_Corr","Nb_Zygo","Is_Correct"] # columns with relevant information from dataframe 
    trial_info = pd.read_csv("Trial_information_narcolepsy.csv", usecols=cols_inc) 
    expected_muscle = ["Corr", "Zygo"]

    epochs = mne.Epochs(raw, events, tmin=-0, tmax=9,baseline=None, detrend=0,
                    reject=None, preload=True, on_missing='warn')

    epochs.metadata = trial_info[(trial_info["Subject"] == subject) 
                                & (trial_info["Trigger"].isin([201.0, 202.0]))
                                & (trial_info["Nap_ID"] == int(block))]


    # adding metadata column for true activation
    # add another column for neither muscle being activated  
    true_activations = [] 
    for i in range(len(epochs.metadata)):
        true_ind = expected_muscle.index(epochs.metadata.iloc[i]["Expected_Muscle"]) 
        if (epochs.metadata.iloc[i]["Is_Correct"] == 0):
            if(epochs.metadata.iloc[i]["Nb_Corr"] < 3 and epochs.metadata.iloc[i]["Nb_Zygo"] < 3):
                true_activations.append("None")
            else:
                true_activations.append(expected_muscle[true_ind-1])
        else:
            true_activations.append(expected_muscle[true_ind])


    epochs.metadata["True_activation"] = true_activations
    return epochs, df_triggers 

In [5]:
# gets features in a sliding window of size window samples with a step size of a certain number of samples 
def get_features(epoch):
     global frq
     global window 
     global step 

     # pad epoch to preserve sample number 
     pad_left  = window // 2
     pad_right = window - 1 - pad_left   
     
     epoch_padded = epoch
     epoch_padded = np.pad(epoch, (pad_left, pad_right), mode="edge")  

     # take sliding window 
     epoch_sw = np.lib.stride_tricks.sliding_window_view(epoch_padded,window)[::step]


     var = np.var(epoch_sw, axis=-1) # calculate variance over window 
     rms =  np.sqrt((1/window)*np.sum(epoch_sw**2, axis=-1)) # calculate rms over window  
     wl = np.sum(np.abs(np.diff(epoch_sw, axis=1)), axis=1) # calculate wl over window  

     # frequency features 
     X = np.fft.rfft(epoch_sw, axis=1)
     PSD = (1/(frq*window)) * np.abs(X)**2
     cumulative = np.cumsum(PSD, axis=1)
     total_power = cumulative[:, -1]
     half_power = total_power / 2

     indices = (cumulative >= half_power[:, None]).argmax(axis=1)
     frequencies = np.fft.rfftfreq(window, 1/frq)

     fmd = frequencies[indices]

  
     return var, rms, wl, fmd


Classification Functions 

In [91]:
# compares power of different measures of muscle activations 
# and returns which muslce was activated based on which has highest power 
def compare_power(zygo,corr):
    zygo_power = np.mean(zygo**2)
    corr_power = np.mean(corr**2)

    zygo_power = np.mean(welch(zygo, frq, nperseg=1024))
    corr_power = np.mean(welch(corr, frq, nperseg=1024))

    print(zygo_power,corr_power,np.abs(zygo_power - corr_power))

    if (np.abs(zygo_power - corr_power) < .04): # hard code threshold for activations to be same (did a relative comparison because other subjects may have different thresholds)
        return "None"
    elif ((zygo_power - corr_power) > 0):  
        return "Zygo"
    else:
        return "Corr"

# determines activation based on power and features -- currently comparing between three and classifying if two match -- only issue is if there are two or so contractions it is not classified as none here 
def determine_activation(zygo,corr,zygo_var,corr_var,zygo_wl,corr_wl):
    # compare power of signal and features 
    sig_power = compare_power(zygo,corr)
    print(sig_power)
    '''
    var_power = compare_power(zygo_var,corr_var)
    wl_power = compare_power(zygo_wl,corr_wl)
  
    if (sig_power == 'None'):
        muscle_activated = "None"
    elif (sig_power == var_power == wl_power):
        muscle_activated = sig_power
    elif sig_power == var_power or sig_power == wl_power:
        muscle_activated = sig_power
    elif var_power == wl_power:
        muscle_activated = var_power
    else:
        muscle_activated = "conflict"
    '''
    return sig_power 


NEED TO PUT THIS IN A COZY FUNCTION AND FIGURE OUT A NICE WAY TO MAKE A BIG DF

In [13]:
def classify_activations(epochs,subject,block):
    # defining data frame for results 
    activation_results_sub = pd.DataFrame(
        index=range(num_epochs),
        columns=[
            "Subject",
            "Nap Number",
            "Triggers_Order_Nap", # epochs 
            "Muscle_Activated_Data",
            "True_Muscle_Activated",
            "Num_Contractions_Zygo_Data",
            "Num_Contractions_Zygo",
            "Num_Contractions_Corr_Data",
            "Num_Contractions_Corr",
            "Contractions_Zygo_Ind",
            "Contractions_Corr_Ind",
            "Activation_Match",
            "Match_Zygo",
            "Match_Corr",
            "Total_Match", # if everything matches (muscle activated and correct contractions)
            "Classification_Match"
        ]
        )   

    muscle_activation_match = [] # for debugging purposes 
    # loop through each of the epochs 
    for t in range(len(epochs)): 

        # extract epoch  
        epoch_zygo = np.squeeze(epochs[t].get_data(picks=['Zygo']))
        epoch_corr = np.squeeze(epochs[t].get_data(picks=['Corr']))

        # calculate some features 
        epoch_zygo_var, epoch_zygo_rms, epoch_zygo_wl, epoch_zygo_fmd = get_features(epoch_zygo)
        epoch_corr_var, epoch_corr_rms, epoch_corr_wl, epoch_corr_fmd = get_features(epoch_corr)

        muscle_activated =  determine_activation(epoch_zygo,epoch_corr,epoch_zygo_var,epoch_corr_var,epoch_zygo_wl,epoch_corr_wl)
        activation_match = epochs[t].metadata['True_activation'].iloc[0]==muscle_activated

        muscle_activation_match.append(activation_match)


        if  (muscle_activated == "None"):
            peaks_zygo, _ = find_peaks(epoch_zygo_var,height=np.mean(epoch_zygo_var), width=30)
            peaks_corr, _ = find_peaks(epoch_corr_var,height=np.mean(epoch_corr_var), width=30)
        else:
            if (muscle_activated == "Zygo"):
                height = np.mean(epoch_zygo_var) 
            elif (muscle_activated == "Corr"):
                height = np.mean(epoch_corr_var)
            peaks_zygo, _ = find_peaks(epoch_zygo_var,height=height, width=30)
            peaks_corr, _ = find_peaks(epoch_corr_var,height=height, width=30)
        

        num_contractions_zygo_data = len(peaks_zygo) # number of contractions from data 
        num_contractions_corr_data = len(peaks_corr) # number of contractions from data 

        match_zygo = num_contractions_zygo_data == epochs.metadata.iloc[t]["Nb_Zygo"]
        match_corr = num_contractions_corr_data == epochs.metadata.iloc[t]["Nb_Corr"]

        # check match 
        if (activation_match):
            if (muscle_activated == "Zygo"):
                match_total = match_zygo
            elif (muscle_activated == "Corr"):
                match_total = match_corr
            elif (muscle_activated == "None"):
                match_total = activation_match
            else:
                match_total = False
        else:
            match_total = False

        # match based on classification 
        if (activation_match):
            if (muscle_activated == "None"):
                classification_match = activation_match
            else:
                if (num_contractions_zygo_data >= 2):
                    classify = "Zygo"
                elif (num_contractions_corr_data >= 2):
                    classify = "Corr"
                else:
                    classify = "None"
                classification_match = (classify == epochs[t].metadata['True_activation'].iloc[0])
                
        else:
            classification_match = False
        
        #print(t, classify)

        # fill dataframe 
        activation_results_sub.loc[t] = [
            subject, 
            int(block),
            t + 1,
            muscle_activated,
            epochs[t].metadata['True_activation'].iloc[0],
            num_contractions_zygo_data,
            epochs.metadata.iloc[t]["Nb_Zygo"],
            num_contractions_corr_data,
            epochs.metadata.iloc[t]["Nb_Corr"],
            peaks_zygo,
            peaks_corr,
            int(activation_match),
            int(match_zygo), 
            int(match_corr),
            int(match_total), 
            int(classification_match)
        ]
    
        
    # indices where the muscle activation determined from activity does not match data 
    conflict_indices_sub = [i+1 for i, val in enumerate(muscle_activation_match) if not val] 
    
    return activation_results_sub, conflict_indices_sub

In [92]:
# initialize parameters for what folder to loop through 
i = 0 
activation_results_mat = []

for root,dirs,files in os.walk(raw_path):
    for file in files:
        if "dpa" not in file and ".DS_Store" not in file: # so only take each subject once 
            subject = file.split(".")[0][:-2]
            block = file.split(".")[0][-2:]

            print(subject+block)

            if (subject == "NL02IF" or subject == "NL05WW" or subject == "NL01SS"):
                continue
            
            subject_epoch, _ = pre_process_subjets(subject,block)
            activation_results_sub, conflict_indices_sub = classify_activations(subject_epoch,subject,block)

            activation_results_mat.append(activation_results_sub)


NL04NF03
31.254927369849863 31.35366310528719 0.09873573543732661
Corr
31.254846907791855 31.462641914003193 0.20779500621133806
Corr
31.256413360467974 31.3799086068032 0.12349524633522435
Corr
31.254206794653125 31.303667444451737 0.0494606497986112
Corr
31.87016090912649 31.293934988051657 0.5762259210748333
Zygo
31.254335889915478 31.36052725168286 0.10619136176738309
Corr
31.257659414045253 31.260015144294236 0.0023557302489827237
None
31.255800391302145 31.318088103410297 0.06228771210815154
Corr
31.267039931683215 31.421113162018546 0.1540732303353316
Corr
31.255877598643337 31.259077600498326 0.003200001854988699
None
31.325078235856125 31.258960352365186 0.06611788349093928
Zygo
31.25676029194569 31.277999840196316 0.021239548250626683
None
31.26226387295174 31.328060668518184 0.06579679556644535
Corr
31.25499403527892 31.32475762742325 0.06976359214433003
Corr
31.254256659100925 31.25759678456304 0.0033401254621168164
None
31.25485223611662 31.297394855122157 0.04254261900553

KeyboardInterrupt: 

In [95]:
activation_results = pd.concat(activation_results_mat, ignore_index=True)
activation_results.to_excel("activation_results_02032026_2.xlsx", index=False)